# 05 · Giải thích mô hình (phụ trách: Khoẻ)

**Đề:** *"Provide patient-level model explanations"* (có trong tên đề: *"with Explainability"*).
Mã nguồn: `trustworthy_ai/explain.py`, `trustworthy_ai/labels.py`, `trustworthy_ai/bundle.py`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

In [ ]:
R = load_results(); sh = R['shap']; print(sh['method'], '| sai số cộng tính:', sh['additivity_max_error'], '| ổn định:', sh['fold_stability'])

## 1. Toàn cục: top đặc trưng và theo nhóm thông tin

In [ ]:
pd.DataFrame(sh['top_features']).head(15)

In [ ]:
pd.DataFrame(sh['families']).head(10)

## 2. Hợp lý lâm sàng và độ trung thực

In [ ]:
pd.DataFrame(sh['plausibility'])

In [ ]:
pd.DataFrame(sh['deletion_curve'])

## 3. Giải thích cho MỘT bệnh nhân (ca demo, nạp gói mô hình như API)

In [ ]:
from trustworthy_ai.bundle import HeartGuardModel
from trustworthy_ai.labels import describe
from backend.app.demo_cases import demo_cases
d = sorted((ROOT / "artifacts/models" / TASK).glob("*/SHA256SUMS"))[-1].parent
m = HeartGuardModel(d)
case = demo_cases(TASK)[2]
r = m.predict(case["features"], top_k=8)[0]
print(case["title"], "→ xác suất đã hiệu chuẩn", r["calibrated_probability"], r["risk_level"])
pd.DataFrame([{**f, **describe(f["feature"])} for f in r["explanation"]["top_features"]])

## 4. Lý do của mô hình thay đổi theo thời gian

In [ ]:
R['explanation_drift']

## Câu hỏi tự kiểm tra
1. SHAP ở thang nào? Vì sao không cộng thẳng ra phần trăm xác suất đã hiệu chuẩn?
2. Vì sao "thiếu sinh hiệu làm giảm nguy cơ" không có nghĩa là thiếu dữ liệu tốt cho sức khỏe?
3. Deletion curve chứng minh điều gì? Tỷ số bao nhiêu là tốt?
4. SHAP có chứng minh quan hệ nhân quả không?